In [ ]:
from loader import load_ds2

import time
import os
from datetime import datetime
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from SHiP.ultrametric_tree import UltrametricTreeType as UTreeType, AVAILABLE_ULTRAMETRIC_TREE_TYPES
from SHiP.partitioning import PartitioningMethod as PMethod, AVAILABLE_PARTITIONING_METHODS
from SHiP import SHiP
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import matplotlib.pyplot as plt
from loader import StudyLogger

study_logger = StudyLogger(log_file="user_study_log2.txt")

In [ ]:
data = load_ds2()
df = pd.DataFrame(data.data, columns=[f"{i}" for i in range(data.data.shape[1])])
y_true = data.target  # ground truth (used for ARI only)
X = df.values
print(f"Shape: {X.shape}, Features: {list(df.columns)}")
df.describe()

In [ ]:
# === CHANGE THESE PARAMETERS ===
tree_type = UTreeType.KDTree        # options: DCTree, HST, CoverTree, KDTree, BallTree, RPTree, ...
power = 2                       # 0=k-center, 1=k-median, 2=k-means
partition_method = PMethod.Elbow    # options: K, Elbow, MedianOfElbows, Stability, ...
config = {"k": 5}                   # used by K, Threshold, QCoverage, etc.
# ================================

study_logger.start_timer()
model = SHiP(data=X, treeType=tree_type, config=config)
labels = model.fit_predict(hierarchy=power, partitioningMethod=partition_method)

# Calculate ARI but don't print it
ari = adjusted_rand_score(y_true, labels)

# Log parameters and ARI
params = {
    "tree_type": tree_type.name,
    "hierarchy": power,
    "partition_method": partition_method.name,
    "config": config
}
study_logger.log(params, ari)

# Display info
print(f"Clusters found: {len(np.unique(labels))}")
print(f"Time elapsed: {study_logger.get_elapsed_time()} (min:sec)")
print(f"Run count: {study_logger.get_run_count()}")


In [ ]:
pca = PCA(n_components=2, random_state=69)
X_2d = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
unique_labels = np.unique(labels)
cmap = plt.cm.tab10
for i, label in enumerate(unique_labels):
    mask = labels == label
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[cmap(i % 10)], label=f"Cluster {label}", s=20, alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title(f"pca — {tree_type.name}, hierarchy={power}, {partition_method.name}")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Get ultrametric distance matrix from SHiP tree, convert to scipy linkage
tree = model.get_tree(hierarchy=power)
dist_matrix = tree.distance_matrix()
np.fill_diagonal(dist_matrix, 0)  # ensure diagonal is exactly zero
condensed = squareform(dist_matrix)
linkage_matrix = linkage(condensed, method="average")

plt.figure(figsize=(12, 5))
dendrogram(linkage_matrix, truncate_mode="level", p=5, no_labels=True)
plt.title("Hierarchy (truncated)")
plt.show()

In [ ]:
sil = silhouette_score(X, labels)
db = davies_bouldin_score(X, labels)
ch = calinski_harabasz_score(X, labels)

print(f"Silhouette Score:       {sil:.3f}  (higher is better, range [-1, 1])")
print(f"Davies-Bouldin Index:   {db:.3f}  (lower is better)")
print(f"Calinski-Harabasz:      {ch:.1f}  (higher is better)")